# GTZAN Music Genre Classification - Sprint 2: PyTorch Data Pipeline

## Prerequisites
This notebook assumes two things have already been run:
1. **`clean_preprocess_dataset.ipynb`** — padded/trimmed every clip to 30s @
   22,050 Hz and extracted 18 mean-pooled numeric features per track
   (tempo, chroma mean, 13 MFCC means, spectral centroid, spectral rolloff,
   zero-crossing rate), saved to `Data_Music/processed/data.csv`.
2. **`Sql_Workflow.ipynb`** — populated the PostgreSQL database and assigned
   a stratified 70/15/15 train/val/test split, queryable via `vw_clean_tracks`.

## Important note on the data shape
The preprocessing step **mean-pools MFCCs over the full clip**, so what we
have per track is a flat vector of 18 scalars — not a time-series or 2D
spectrogram. That changes two of the usual audio-pipeline steps:

- **"Pad/trim to standard length" and "extract MFCCs as a tensor"** are
  already satisfied by the preprocessing step — there's no raw-audio work
  left to do here, just loading the resulting feature table and turning
  each row into a tensor.
- **"Time shifting" and "frequency masking"** don't have a literal
  equivalent anymore since the time axis was averaged away. This notebook
  substitutes tabular-appropriate stand-ins instead:
  - **Feature noise jitter** (Gaussian noise on the normalized vector) in
    place of waveform-level noise injection.
  - **Random feature masking** (zeroing a random subset of the MFCC /
    spectral columns) in place of frequency masking.
  - No time-shift equivalent is applied, since there's no time axis left.

  If true temporal augmentation (time-shift, real frequency masking) turns
  out to matter for model performance, that would require re-extracting
  full MFCC *sequences* (not means) in the preprocessing step — happy to
  help build that version if you want to compare.

- Downstream, since this is now a flat feature vector rather than a 2D
  grid, a small **MLP** is the natural model architecture — not a CNN.

## Workflow
1. → Imports
2. → Load DB credentials + connect
3. → Pull split assignments from `vw_clean_tracks`
4. → Load extracted features from `data.csv`
5. → Merge features with split assignments
6. → Define feature columns + label map
7. → Normalize (z-score, fit on train split only)
8. → Augmentation functions (train split only)
9. → Custom `GTZANFeatureDataset` class
10. → Build datasets + DataLoaders
11. → Sanity-check a sample batch

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

import torch
from torch.utils.data import Dataset, DataLoader

torch.set_num_threads(os.cpu_count())

print('✓ Libraries loaded')
print(f'✓ PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()} | CPU threads: {torch.get_num_threads()}')

✓ Libraries loaded
✓ PyTorch 2.10.0 | CUDA available: False | CPU threads: 12


## Load DB credentials and connect

Same `.env` / connection pattern as `Sql_Workflow.ipynb`.

In [2]:
load_dotenv()

DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST     = 'localhost'
DB_PORT     = 5432
DB_NAME     = 'music_genre_db'

assert DB_USER,     'DB_USER not found in .env'
assert DB_PASSWORD, 'DB_PASSWORD not found in .env'

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

with engine.connect() as conn:
    db_info = conn.execute(text('SELECT current_database(), current_user;')).fetchone()
    print(f'✓ Connected to: {db_info[0]} as {db_info[1]}')

✓ Connected to: music_genre_db as postgres


## Pull split assignments from `vw_clean_tracks`

This gives us the train/val/test split and genre label for every clean
(non-corrupted, non-duplicate) track. We derive `filename` from `file_path`
so we can join it against the feature CSV in the next step.

In [3]:
df_split = pd.read_sql(
    'SELECT file_path, label, split FROM vw_clean_tracks', engine
)

# Derive filename from file_path (handles both '/' and '\\' separators)
df_split['filename'] = (
    df_split['file_path'].str.replace('\\', '/', regex=False).str.split('/').str[-1]
)

print(f'✓ {len(df_split)} clean tracks with split assignments')
print(df_split['split'].value_counts())

✓ 971 clean tracks with split assignments
split
train    677
test     153
val      141
Name: count, dtype: int64


## Load extracted features from `data.csv`

Produced by `clean_preprocess_dataset.ipynb`.

In [4]:
PROCESSED_DIR = Path('../Data_Music/processed')
FEATURES_CSV = PROCESSED_DIR / 'data.csv'

assert FEATURES_CSV.exists(), f'Features CSV not found at {FEATURES_CSV.absolute()} — run clean_preprocess_dataset.ipynb first.'

df_features = pd.read_csv(FEATURES_CSV)
print(f'✓ Loaded {len(df_features)} rows, {df_features.shape[1]} columns from {FEATURES_CSV}')
df_features.head()

✓ Loaded 999 rows, 24 columns from ..\Data_Music\processed\data.csv


,genre,filename,path,duration,sample_rate,channels,tempo,chroma_mean,mfcc1_mean,mfcc2_mean,...,mfcc7_mean,mfcc8_mean,mfcc9_mean,mfcc10_mean,mfcc11_mean,mfcc12_mean,mfcc13_mean,spectral_centroid,spectral_rolloff,zero_crossing_rate
0,blues,blues.00000.wav,..\Data_Music\processed\blues\blues.00000.wav,30.0,22050,1,123.046875,0.349951,-113.619385,121.55302,...,-13.692060,15.339378,-12.283618,10.973775,-8.322410,8.806788,-3.665802,1784.416546,3806.418650,0.083066
1,blues,blues.00001.wav,..\Data_Music\processed\blues\blues.00001.wav,30.0,22050,1,67.999589,0.340945,-207.581510,123.99715,...,-8.555368,23.355938,-10.101037,11.906444,-5.558123,5.375942,-2.237833,1529.871314,3548.986873,0.056044
2,blues,blues.00002.wav,..\Data_Music\processed\blues\blues.00002.wav,30.0,22050,1,161.499023,0.363562,-90.776344,140.44861,...,-13.644712,11.623112,-11.775921,9.700466,-13.115349,5.785763,-8.899733,1552.637786,3041.089944,0.076301
3,blues,blues.00003.wav,..\Data_Music\processed\blues\blues.00003.wav,30.0,22050,1,63.024009,0.404848,-199.462000,150.09474,...,-4.828873,9.297849,-0.753142,8.147393,-3.195236,6.085354,-2.476188,1070.110059,2185.061787,0.033309
4,blues,blues.00004.wav,..\Data_Music\processed\blues\blues.00004.wav,30.0,22050,1,135.999178,0.308598,-160.291850,126.19576,...,-23.357162,0.500523,-11.804770,1.203878,-13.085074,-2.809849,-6.935620,1835.507008,3581.003346,0.101500


## Merge features with split assignments

Joined on `filename`, which is unique per track in the GTZAN naming
convention (e.g. `blues.00000.wav`). Tracks flagged corrupted/duplicate in
the database are automatically excluded since they're not in `df_split`.

In [5]:
df = df_features.merge(
    df_split[['filename', 'label', 'split']],
    on='filename',
    how='inner'
)

print(f'Features rows:        {len(df_features)}')
print(f'Clean split rows:     {len(df_split)}')
print(f'Merged (usable) rows: {len(df)}')
print()
print(df['split'].value_counts())

# Flag anything that didn't merge, for visibility
unmatched = set(df_features['filename']) - set(df_split['filename'])
if unmatched:
    print(f'\n⚠ {len(unmatched)} feature rows had no matching clean split assignment (excluded): {sorted(unmatched)[:5]}...')

Features rows:        999
Clean split rows:     971
Merged (usable) rows: 971

split
train    677
test     153
val      141
Name: count, dtype: int64

⚠ 28 feature rows had no matching clean split assignment (excluded): ['disco.00098.wav', 'disco.00099.wav', 'hiphop.00039.wav', 'hiphop.00045.wav', 'hiphop.00076.wav']...


## Define feature columns and label map

We use the 18 signal-derived features (tempo, chroma, 13 MFCC means,
spectral centroid/rolloff, zero-crossing rate) and drop the metadata
columns (`duration`, `sample_rate`, `channels`) since those are constant
across the cleaned dataset and carry no predictive signal.

In [6]:
FEATURE_COLS = [
    'tempo', 'chroma_mean',
    'mfcc1_mean', 'mfcc2_mean', 'mfcc3_mean', 'mfcc4_mean', 'mfcc5_mean',
    'mfcc6_mean', 'mfcc7_mean', 'mfcc8_mean', 'mfcc9_mean', 'mfcc10_mean',
    'mfcc11_mean', 'mfcc12_mean', 'mfcc13_mean',
    'spectral_centroid', 'spectral_rolloff', 'zero_crossing_rate',
]

assert all(c in df.columns for c in FEATURE_COLS), 'Missing expected feature column(s)'

genre_list = sorted(df['label'].unique())
label_map = {name: i for i, name in enumerate(genre_list)}

df_train = df[df['split'] == 'train'].reset_index(drop=True)
df_val   = df[df['split'] == 'val'].reset_index(drop=True)
df_test  = df[df['split'] == 'test'].reset_index(drop=True)

print(f'Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print(f'Label map: {label_map}')
print(f'Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')

Feature columns (18): ['tempo', 'chroma_mean', 'mfcc1_mean', 'mfcc2_mean', 'mfcc3_mean', 'mfcc4_mean', 'mfcc5_mean', 'mfcc6_mean', 'mfcc7_mean', 'mfcc8_mean', 'mfcc9_mean', 'mfcc10_mean', 'mfcc11_mean', 'mfcc12_mean', 'mfcc13_mean', 'spectral_centroid', 'spectral_rolloff', 'zero_crossing_rate']
Label map: {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}
Train: 677 | Val: 141 | Test: 153


## Normalize feature values across the dataset

Z-score normalization using mean/std computed **from the train split
only**, then reused for val/test — this avoids leaking val/test statistics
into training. Since the features are already scalar (no per-file audio
loading needed), this is a simple column-wise operation.

In [7]:
train_mean = df_train[FEATURE_COLS].mean().values.astype(np.float32)
train_std  = (df_train[FEATURE_COLS].std().values.astype(np.float32) + 1e-8)

print('✓ Normalization stats computed from train split')
print(pd.DataFrame({'feature': FEATURE_COLS, 'mean': train_mean, 'std': train_std}))

✓ Normalization stats computed from train split
               feature         mean          std
0                tempo   120.781418    29.232414
1          chroma_mean     0.376392     0.082768
2           mfcc1_mean  -147.487579   102.334740
3           mfcc2_mean    99.885475    30.951189
4           mfcc3_mean    -8.710272    20.956207
5           mfcc4_mean    36.079411    16.459309
6           mfcc5_mean    -1.181956    12.068984
7           mfcc6_mean    14.682989    11.857622
8           mfcc7_mean    -5.245478    10.035619
9           mfcc8_mean    10.211465    10.430092
10          mfcc9_mean    -7.125666     8.394412
11         mfcc10_mean     7.670831     7.908211
12         mfcc11_mean    -6.039727     6.738326
13         mfcc12_mean     4.418519     6.717247
14         mfcc13_mean    -4.665586     6.037592
15   spectral_centroid  2189.229492   708.750488
16    spectral_rolloff  4545.930176  1555.112183
17  zero_crossing_rate     0.102943     0.041976


## Augmentation (train split only)

Tabular stand-ins for the usual waveform/spectrogram augmentations, since
the time axis was averaged away during preprocessing:

- **`add_feature_noise`** — small Gaussian jitter on the normalized vector,
  standing in for background-noise augmentation.
- **`random_feature_mask`** — zeroes out a random subset of the MFCC /
  spectral columns, standing in for frequency masking.

Both operate in normalized (z-scored) space and are applied only when the
`Dataset` is constructed with `augment=True` (train split).

In [8]:
# Indices of the maskable columns (chroma + 13 MFCCs + 3 spectral features;
# tempo is left alone since it's a single global descriptor, not a "band")
MASKABLE_START = FEATURE_COLS.index('chroma_mean')
MASKABLE_END   = len(FEATURE_COLS)   # exclusive


def add_feature_noise(x, noise_std=0.1):
    """Add small Gaussian jitter to a normalized feature vector."""
    noise = np.random.randn(*x.shape).astype(np.float32) * noise_std
    return x + noise


def random_feature_mask(x, mask_prob=0.15):
    """Randomly zero out a subset of the maskable (chroma/MFCC/spectral) columns."""
    x = x.copy()
    for i in range(MASKABLE_START, MASKABLE_END):
        if random.random() < mask_prob:
            x[i] = 0.0
    return x


def augment_features(x):
    if random.random() < 0.5:
        x = add_feature_noise(x)
    if random.random() < 0.5:
        x = random_feature_mask(x)
    return x

print('✓ Augmentation functions defined: add_feature_noise, random_feature_mask')

✓ Augmentation functions defined: add_feature_noise, random_feature_mask


## Custom PyTorch `Dataset`

`GTZANFeatureDataset` takes a dataframe of pre-extracted feature rows,
normalizes each row with the train-set stats, optionally augments (train
only), and returns a `(feature_tensor, label_tensor)` pair. Since there's
no audio file to load at `__getitem__` time, this is fast — no disk I/O or
`librosa` calls happen here.

In [9]:
class GTZANFeatureDataset(Dataset):
    """PyTorch Dataset over pre-extracted GTZAN tabular audio features."""

    def __init__(self, dataframe, feature_cols, label_map, mean, std, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.label_map = label_map
        self.mean = mean
        self.std = std
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        x = row[self.feature_cols].values.astype(np.float32)
        x = (x - self.mean) / self.std

        if self.augment:
            x = augment_features(x)

        feature_tensor = torch.from_numpy(x).float()
        label_tensor = torch.tensor(self.label_map[row['label']], dtype=torch.long)

        return feature_tensor, label_tensor

print('✓ GTZANFeatureDataset class defined')

✓ GTZANFeatureDataset class defined


## Build the datasets and DataLoaders

Augmentation is enabled only for the train `Dataset`. `num_workers=0` is
used since this runs inside a Jupyter kernel on Windows — and with no audio
I/O left in `__getitem__`, extra workers wouldn't buy much here anyway.

In [10]:
BATCH_SIZE = 32

train_dataset = GTZANFeatureDataset(df_train, FEATURE_COLS, label_map, train_mean, train_std, augment=True)
val_dataset   = GTZANFeatureDataset(df_val,   FEATURE_COLS, label_map, train_mean, train_std, augment=False)
test_dataset  = GTZANFeatureDataset(df_test,  FEATURE_COLS, label_map, train_mean, train_std, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'✓ DataLoaders ready — train batches: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}')

✓ DataLoaders ready — train batches: 22, val: 5, test: 5


## Confirm a sample batch loads correctly

Checks shape (`[batch, num_features]`), dtype, and that normalized values
fall in a sane range (roughly centered on 0, since they're z-scored).

In [11]:
batch_features, batch_labels = next(iter(train_loader))

print('=== SAMPLE TRAIN BATCH ===\n')
print(f'Feature batch shape: {tuple(batch_features.shape)}  (batch, num_features)')
print(f'Label batch shape:   {tuple(batch_labels.shape)}')
print(f'Feature dtype:       {batch_features.dtype}')
print(f'Label dtype:         {batch_labels.dtype}')
print(f'Value range:         [{batch_features.min().item():.3f}, {batch_features.max().item():.3f}]')
print(f'Value mean/std:      {batch_features.mean().item():.3f} / {batch_features.std().item():.3f}')

inv_label_map = {v: k for k, v in label_map.items()}
print(f'Genres in batch:     {sorted({inv_label_map[l] for l in batch_labels.tolist()})}')

=== SAMPLE TRAIN BATCH ===

Feature batch shape: (32, 18)  (batch, num_features)
Label batch shape:   (32,)
Feature dtype:       torch.float32
Label dtype:         torch.int64
Value range:         [-3.275, 4.090]
Value mean/std:      0.016 / 1.026
Genres in batch:     ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
